# Project 83 — Local Receipt Intelligence
## Parse Receipts → Categorize → Budget Analysis

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Receipt Corpus

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import pandas as pd, json

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

receipts = [
    "Whole Foods Market 03/01/2025 Organic Apples $5.99 Almond Milk $4.49 "
    "Quinoa $6.99 Chicken Breast $12.49 Total: $29.96",
    "Shell Gas Station 03/02/2025 Regular Unleaded 12.5gal @$3.29 = $41.13 "
    "Car Wash $8.00 Total: $49.13",
    "Amazon.com Order 03/03/2025 Wireless Mouse $24.99 USB Hub $15.99 "
    "HDMI Cable $9.99 Shipping: Free Total: $50.97",
    "Starbucks 03/04/2025 Grande Latte $5.75 Croissant $3.95 Total: $9.70",
    "CVS Pharmacy 03/05/2025 Ibuprofen $8.99 Vitamins $12.49 "
    "Band-Aids $4.99 Total: $26.47",
    "Netflix 03/01/2025 Monthly subscription Standard plan Total: $15.49",
    "Con Edison 03/01/2025 Electricity February 2025 Usage: 650kWh Total: $98.50",
    "Planet Fitness 03/01/2025 Monthly membership Classic Total: $10.00",
]
print(f"Receipts to process: {len(receipts)}")

## Step 2 — Extract & Categorize

In [ ]:
class ReceiptItem(BaseModel):
    name: str
    price: float

class Receipt(BaseModel):
    merchant: str
    date: str
    category: str = Field(description="groceries, gas, electronics, food_drink, health, subscription, utilities, fitness")
    items: list[ReceiptItem]
    total: float
    payment_type: str = Field(default="unknown", description="cash, credit, debit, unknown")

extractor = llm.with_structured_output(Receipt)

parsed = []
for text in receipts:
    r = extractor.invoke(f"Extract receipt data:\n{text}")
    parsed.append(r)
    print(f"  {r.merchant:<25} {r.category:<15} ${r.total:>8.2f}  ({len(r.items)} items)")

## Step 3 — Budget Analysis

In [ ]:
df = pd.DataFrame([r.model_dump() for r in parsed])
df["items_count"] = df["items"].apply(len)

print("SPENDING SUMMARY")
print("=" * 50)
by_cat = df.groupby("category")["total"].agg(["sum", "count", "mean"]).round(2)
by_cat.columns = ["total_spent", "transactions", "avg_per_txn"]
by_cat = by_cat.sort_values("total_spent", ascending=False)
print(by_cat.to_string())
print(f"\nGrand total: ${df['total'].sum():.2f}")
print(f"Average transaction: ${df['total'].mean():.2f}")

# Budget limits
budgets = {"groceries": 200, "food_drink": 50, "subscription": 30, "gas": 100, "electronics": 100}
print("\nBUDGET CHECK:")
for cat, limit in budgets.items():
    spent = df[df["category"] == cat]["total"].sum()
    pct = spent / limit * 100 if limit > 0 else 0
    status = "✓ OK" if spent <= limit else "⚠ OVER"
    print(f"  {cat:<15} ${spent:>8.2f} / ${limit:.2f} ({pct:.0f}%) {status}")

## Step 4 — Spending Insights

In [ ]:
insight_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a personal finance advisor. Analyze spending and give 3 actionable tips."),
    ("human", "Spending data:\n{data}\n\nBudgets:\n{budgets}\n\nGive 3 specific tips:")
])
insight_chain = insight_prompt | llm | StrOutputParser()

tips = insight_chain.invoke({
    "data": by_cat.to_string(),
    "budgets": json.dumps(budgets),
})
print("SPENDING INSIGHTS")
print("=" * 50)
print(tips[:600])

## What You Learned
- **Receipt parsing** with structured extraction
- **Auto-categorization** of expenses
- **Budget tracking** with variance analysis
- **AI spending insights** for personal finance